# Olist Analytics Pro - Data Profiling

**Project:** Enterprise E-Commerce Intelligence Platform  
**Phase:** 1 - Data Profiling and Quality Assessment  
**Author:** Portfolio Project  
**Date:** 2026-10-20

---

## Objectives

In this notebook, we will:
1. Load and inspect all Olist CSV files
2. Profile data structure (rows, columns, data types)
3. Identify missing values and duplicates
4. Analyze ID columns for primary/foreign key candidates
5. Generate statistical summaries for numeric columns
6. Analyze date ranges and categorical distributions
7. Document data quality issues
8. Create a comprehensive data dictionary

---

## Business Context

Before analyzing e-commerce performance, we must understand:
- What data is available?
- What is the quality of the data?
- How are tables related?
- What issues need to be addressed?

This systematic profiling approach demonstrates professional data analyst methodology.

---
## Step 1: Import Libraries and Setup

We'll use **Pandas** for data manipulation and **NumPy** for numerical operations.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Configure display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("[SUCCESS] Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

---
## Step 2: Define Data Directory and Discover Files

Using **Path** from pathlib for cross-platform compatibility.

In [ ]:
# Define project paths
project_root = Path.cwd().parent
data_dir = project_root / 'dataset'
reports_dir = project_root / 'reports'

# Create reports directory if it doesn't exist
reports_dir.mkdir(parents=True, exist_ok=True)

print(f"Project Root: {project_root}")
print(f"Data Directory: {data_dir}")
print(f"Reports Directory: {reports_dir}")

In [ ]:
# Discover all CSV files in the dataset directory
all_csv_files = sorted(data_dir.glob('*.csv'))

# Filter for Olist-related files
olist_files = [
    f for f in all_csv_files 
    if 'olist' in f.name.lower() or 'product_category' in f.name.lower()
]

print(f"\n[INFO] Found {len(olist_files)} Olist data files:\n")
for i, file in enumerate(olist_files, 1):
    file_size_mb = file.stat().st_size / (1024 * 1024)
    print(f"{i}. {file.name:45s} ({file_size_mb:.2f} MB)")

---
## Step 3: Load and Profile Each Dataset

We'll create a comprehensive profiling summary for each file using **vectorized Pandas operations**.

### 3.1 Customers Dataset

In [ ]:
# Load customers dataset
customers = pd.read_csv(data_dir / 'olist_customers_dataset.csv')

print("="*80)
print("CUSTOMERS DATASET PROFILE")
print("="*80)

# Basic shape information
print(f"\n[DATA] Shape: {customers.shape[0]:,} rows x {customers.shape[1]} columns")
print(f"[DATA] Memory Usage: {customers.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

# Display first few rows
print("\n[SAMPLE] First 3 rows:")
customers.head(3)

In [ ]:
# Data types and column info
print("[INFO] Column Information:")
customers.info()

In [ ]:
# Missing values analysis using vectorized operations
missing_counts = customers.isnull().sum()
missing_pct = (missing_counts / len(customers) * 100).round(2)

# Create missing values summary DataFrame
missing_summary = pd.DataFrame({
    'Column': customers.columns,
    'Missing_Count': missing_counts.values,
    'Missing_Percentage': missing_pct.values,
    'Data_Type': customers.dtypes.values
})

print("\n[ANALYSIS] Missing Values Summary:")
print(missing_summary.to_string(index=False))

# Check if any columns have missing values
if missing_counts.sum() == 0:
    print("\n[SUCCESS] No missing values detected")
else:
    print(f"\n[WARNING] {missing_counts[missing_counts > 0].count()} columns have missing values")

In [ ]:
# Duplicate rows analysis using vectorized operation
duplicate_mask = customers.duplicated()
duplicate_count = duplicate_mask.sum()
duplicate_pct = (duplicate_count / len(customers) * 100).round(2)

print(f"\n[ANALYSIS] Duplicate Rows: {duplicate_count:,} ({duplicate_pct}%)")

if duplicate_count == 0:
    print("[SUCCESS] No duplicate rows")
else:
    print("[WARNING] Duplicate rows detected")

In [ ]:
# ID column analysis - identify columns with 'id' in name using vectorized string operation
id_columns = customers.columns[customers.columns.str.contains('id', case=False)]

print(f"\n[ANALYSIS] ID Column Analysis ({len(id_columns)} columns):")
print()

# Vectorized nunique() operation for all ID columns at once
unique_counts = customers[id_columns].nunique()
null_counts = customers[id_columns].isnull().sum()

# Create ID analysis DataFrame
id_analysis = pd.DataFrame({
    'Column': id_columns,
    'Total_Rows': len(customers),
    'Unique_Values': unique_counts.values,
    'Duplicate_Count': len(customers) - unique_counts.values,
    'Null_Count': null_counts.values,
    'Cardinality_Ratio': (unique_counts.values / len(customers) * 100).round(2)
})

# Determine if column is a potential primary key
id_analysis['Potential_Primary_Key'] = (
    (id_analysis['Duplicate_Count'] == 0) & 
    (id_analysis['Null_Count'] == 0)
)

print(id_analysis.to_string(index=False))

# Key insights
print("\n[INSIGHT] Key Findings:")
for _, row in id_analysis.iterrows():
    if row['Potential_Primary_Key']:
        print(f"  - {row['Column']}: UNIQUE - Suitable as Primary Key")
    else:
        print(f"  - {row['Column']}: {row['Duplicate_Count']:,} duplicates - Indicates repeat customers")

In [ ]:
# Cardinality analysis for all columns using vectorized operations
cardinality = customers.nunique()
cardinality_pct = (cardinality / len(customers) * 100).round(2)

cardinality_summary = pd.DataFrame({
    'Column': customers.columns,
    'Unique_Values': cardinality.values,
    'Cardinality_Pct': cardinality_pct.values,
    'Data_Type': customers.dtypes.values
}).sort_values('Cardinality_Pct', ascending=False)

print("\n[ANALYSIS] Cardinality Analysis (sorted by uniqueness):")
print(cardinality_summary.to_string(index=False))

In [ ]:
# Categorical column analysis - select object dtype columns
categorical_cols = customers.select_dtypes(include=['object']).columns
# Exclude ID columns from categorical analysis
categorical_cols = categorical_cols[~categorical_cols.str.contains('id', case=False)]

print(f"\n[ANALYSIS] Categorical Columns ({len(categorical_cols)} columns):\n")

for col in categorical_cols:
    # Vectorized value_counts() operation
    value_counts = customers[col].value_counts()
    
    print(f"\n{col}:")
    print(f"  Unique values: {len(value_counts)}")
    print(f"  Most common: {value_counts.index[0]} (n={value_counts.iloc[0]:,}, {value_counts.iloc[0]/len(customers)*100:.1f}%)")
    
    # Show top 5 for low cardinality columns
    if len(value_counts) <= 50:
        print(f"\n  Top 5 values:")
        print(value_counts.head(5).to_frame('Count'))

### 3.2 Orders Dataset

In [ ]:
# Load orders dataset
orders = pd.read_csv(data_dir / 'olist_orders_dataset.csv')

print("="*80)
print("ORDERS DATASET PROFILE")
print("="*80)

print(f"\n[DATA] Shape: {orders.shape[0]:,} rows x {orders.shape[1]} columns")
print(f"[DATA] Memory Usage: {orders.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

print("\n[SAMPLE] First 3 rows:")
orders.head(3)

In [ ]:
# Missing values analysis
missing_counts = orders.isnull().sum()
missing_pct = (missing_counts / len(orders) * 100).round(2)

missing_summary = pd.DataFrame({
    'Column': orders.columns,
    'Missing_Count': missing_counts.values,
    'Missing_Percentage': missing_pct.values,
    'Data_Type': orders.dtypes.astype(str).values
})

print("\n[ANALYSIS] Missing Values Summary:")
print(missing_summary.to_string(index=False))

# Highlight columns with missing values
cols_with_missing = missing_summary[missing_summary['Missing_Count'] > 0]
if len(cols_with_missing) > 0:
    print(f"\n[WARNING] {len(cols_with_missing)} columns have missing values")
else:
    print("\n[SUCCESS] No missing values")

In [ ]:
# ID column analysis
id_columns = orders.columns[orders.columns.str.contains('id', case=False)]

unique_counts = orders[id_columns].nunique()
null_counts = orders[id_columns].isnull().sum()

id_analysis = pd.DataFrame({
    'Column': id_columns,
    'Total_Rows': len(orders),
    'Unique_Values': unique_counts.values,
    'Duplicate_Count': len(orders) - unique_counts.values,
    'Null_Count': null_counts.values,
    'Cardinality_Ratio': (unique_counts.values / len(orders) * 100).round(2),
    'Potential_Primary_Key': (
        (unique_counts.values == len(orders)) & 
        (null_counts.values == 0)
    )
})

print("\n[ANALYSIS] ID Column Analysis:")
print(id_analysis.to_string(index=False))

In [ ]:
# Date column analysis - identify date columns
date_columns = orders.columns[orders.columns.str.contains('date|timestamp', case=False)]

print(f"\n[ANALYSIS] Date Columns ({len(date_columns)} columns):\n")

date_analysis_results = []

for col in date_columns:
    # Vectorized date parsing
    date_series = pd.to_datetime(orders[col], errors='coerce')
    
    # Vectorized validity check
    valid_mask = date_series.notna()
    valid_count = valid_mask.sum()
    invalid_count = (~valid_mask).sum()
    
    if valid_count > 0:
        min_date = date_series.min()
        max_date = date_series.max()
        date_range_days = (max_date - min_date).days
        
        date_analysis_results.append({
            'Column': col,
            'Valid_Dates': valid_count,
            'Invalid_Dates': invalid_count,
            'Invalid_Pct': round(invalid_count / len(orders) * 100, 2),
            'Min_Date': min_date.date(),
            'Max_Date': max_date.date(),
            'Date_Range_Days': date_range_days
        })

date_analysis_df = pd.DataFrame(date_analysis_results)
print(date_analysis_df.to_string(index=False))

In [ ]:
# Order status analysis - categorical distribution
status_counts = orders['order_status'].value_counts()
status_pct = (status_counts / len(orders) * 100).round(2)

status_summary = pd.DataFrame({
    'Order_Status': status_counts.index,
    'Count': status_counts.values,
    'Percentage': status_pct.values
})

print("\n[ANALYSIS] Order Status Distribution:")
print(status_summary.to_string(index=False))

# Visualize order status
plt.figure(figsize=(10, 6))
status_counts.plot(kind='barh', color='steelblue')
plt.title('Order Status Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Number of Orders')
plt.ylabel('Status')
plt.tight_layout()
plt.show()

print(f"\n[INSIGHT] {status_counts.iloc[0]:,} orders ({status_pct.iloc[0]}%) are '{status_counts.index[0]}'")

### 3.3 Order Items Dataset

In [ ]:
# Load order items dataset
order_items = pd.read_csv(data_dir / 'olist_order_items_dataset.csv')

print("="*80)
print("ORDER ITEMS DATASET PROFILE")
print("="*80)

print(f"\n[DATA] Shape: {order_items.shape[0]:,} rows x {order_items.shape[1]} columns")
print(f"[DATA] Memory Usage: {order_items.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

print("\n[SAMPLE] First 5 rows:")
order_items.head(5)

In [ ]:
# Basic statistics using vectorized describe()
print("\n[ANALYSIS] Numeric Column Statistics:")
order_items.describe()

In [ ]:
# Numeric column deep analysis
numeric_cols = order_items.select_dtypes(include=[np.number]).columns

print(f"\n[ANALYSIS] Detailed Numeric Analysis ({len(numeric_cols)} columns):\n")

# Vectorized statistical calculations
numeric_stats = pd.DataFrame({
    'Column': numeric_cols,
    'Count': order_items[numeric_cols].count().values,
    'Mean': order_items[numeric_cols].mean().values,
    'Median': order_items[numeric_cols].median().values,
    'Std': order_items[numeric_cols].std().values,
    'Min': order_items[numeric_cols].min().values,
    'Max': order_items[numeric_cols].max().values,
    'Skewness': order_items[numeric_cols].skew().values,
    'Zeros': (order_items[numeric_cols] == 0).sum().values,
    'Negatives': (order_items[numeric_cols] < 0).sum().values
})

print(numeric_stats.to_string(index=False))

In [ ]:
# Price and freight analysis
print("\n[ANALYSIS] Price and Freight Analysis:\n")

# Total order value calculation using vectorized operations
order_items['total_value'] = order_items['price'] + order_items['freight_value']

print(f"Price Statistics:")
print(f"  Mean: R$ {order_items['price'].mean():.2f}")
print(f"  Median: R$ {order_items['price'].median():.2f}")
print(f"  Min: R$ {order_items['price'].min():.2f}")
print(f"  Max: R$ {order_items['price'].max():.2f}")
print(f"  Total GMV: R$ {order_items['price'].sum():,.2f}")

print(f"\nFreight Statistics:")
print(f"  Mean: R$ {order_items['freight_value'].mean():.2f}")
print(f"  Median: R$ {order_items['freight_value'].median():.2f}")
print(f"  Free shipping (0.00): {(order_items['freight_value'] == 0).sum():,} items")

print(f"\nTotal Order Value:")
print(f"  Mean: R$ {order_items['total_value'].mean():.2f}")
print(f"  Total: R$ {order_items['total_value'].sum():,.2f}")

In [ ]:
# Multi-item order analysis
# Count items per order using vectorized groupby
items_per_order = order_items.groupby('order_id').size()

print("\n[ANALYSIS] Items Per Order Distribution:\n")
print(f"Total orders: {items_per_order.count():,}")
print(f"Total items: {order_items.shape[0]:,}")
print(f"Average items per order: {items_per_order.mean():.2f}")
print(f"Orders with 1 item: {(items_per_order == 1).sum():,} ({(items_per_order == 1).sum()/len(items_per_order)*100:.1f}%)")
print(f"Orders with 2+ items: {(items_per_order >= 2).sum():,} ({(items_per_order >= 2).sum()/len(items_per_order)*100:.1f}%)")
print(f"Max items in single order: {items_per_order.max()}")

# Distribution of items per order
items_dist = items_per_order.value_counts().sort_index()
print("\nDistribution (top 10):")
print(items_dist.head(10))

### 3.4 Payments Dataset

In [ ]:
# Load payments dataset
payments = pd.read_csv(data_dir / 'olist_order_payments_dataset.csv')

print("="*80)
print("PAYMENTS DATASET PROFILE")
print("="*80)

print(f"\n[DATA] Shape: {payments.shape[0]:,} rows x {payments.shape[1]} columns")

payments.head(5)

In [ ]:
# Payment method analysis using vectorized value_counts
payment_method_counts = payments['payment_type'].value_counts()
payment_method_pct = (payment_method_counts / len(payments) * 100).round(2)

payment_summary = pd.DataFrame({
    'Payment_Method': payment_method_counts.index,
    'Count': payment_method_counts.values,
    'Percentage': payment_method_pct.values
})

print("\n[ANALYSIS] Payment Method Distribution:")
print(payment_summary.to_string(index=False))

# Visualize payment methods
plt.figure(figsize=(10, 6))
payment_method_counts.plot(kind='bar', color='darkgreen')
plt.title('Payment Method Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Payment Method')
plt.ylabel('Number of Payments')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Payment installment analysis
installment_stats = payments.groupby('payment_type')['payment_installments'].agg([
    ('Mean_Installments', 'mean'),
    ('Median_Installments', 'median'),
    ('Max_Installments', 'max')
]).round(2)

print("\n[ANALYSIS] Installment Statistics by Payment Method:")
print(installment_stats)

# Overall installment distribution
installment_dist = payments['payment_installments'].value_counts().sort_index().head(10)
print("\n[ANALYSIS] Top 10 Installment Terms:")
print(installment_dist)

In [ ]:
# Multi-payment order analysis
payments_per_order = payments.groupby('order_id').size()

print("\n[ANALYSIS] Payments Per Order:")
print(f"Orders with 1 payment: {(payments_per_order == 1).sum():,}")
print(f"Orders with 2+ payments: {(payments_per_order >= 2).sum():,}")
print(f"Multi-payment rate: {(payments_per_order >= 2).sum() / len(payments_per_order) * 100:.2f}%")

### 3.5 Reviews Dataset

In [ ]:
# Load reviews dataset
reviews = pd.read_csv(data_dir / 'olist_order_reviews_dataset.csv')

print("="*80)
print("REVIEWS DATASET PROFILE")
print("="*80)

print(f"\n[DATA] Shape: {reviews.shape[0]:,} rows x {reviews.shape[1]} columns")

reviews.head(5)

In [ ]:
# Review score distribution analysis
score_counts = reviews['review_score'].value_counts().sort_index()
score_pct = (score_counts / len(reviews) * 100).round(2)

score_summary = pd.DataFrame({
    'Review_Score': score_counts.index,
    'Count': score_counts.values,
    'Percentage': score_pct.values
})

print("\n[ANALYSIS] Review Score Distribution:")
print(score_summary.to_string(index=False))

print(f"\nMean review score: {reviews['review_score'].mean():.2f}")
print(f"Median review score: {reviews['review_score'].median():.0f}")

# Visualize review scores
plt.figure(figsize=(10, 6))
score_counts.plot(kind='bar', color='coral')
plt.title('Customer Review Score Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Review Score (1-5)')
plt.ylabel('Number of Reviews')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Review comment analysis
title_missing = reviews['review_comment_title'].isnull().sum()
message_missing = reviews['review_comment_message'].isnull().sum()

print("\n[ANALYSIS] Review Comment Completeness:")
print(f"Reviews with title: {len(reviews) - title_missing:,} ({(len(reviews) - title_missing)/len(reviews)*100:.1f}%)")
print(f"Reviews without title: {title_missing:,} ({title_missing/len(reviews)*100:.1f}%)")
print(f"\nReviews with message: {len(reviews) - message_missing:,} ({(len(reviews) - message_missing)/len(reviews)*100:.1f}%)")
print(f"Reviews without message: {message_missing:,} ({message_missing/len(reviews)*100:.1f}%)")

### 3.6 Products Dataset

In [ ]:
# Load products dataset
products = pd.read_csv(data_dir / 'olist_products_dataset.csv')

print("="*80)
print("PRODUCTS DATASET PROFILE")
print("="*80)

print(f"\n[DATA] Shape: {products.shape[0]:,} rows x {products.shape[1]} columns")

products.head(5)

In [ ]:
# Missing values analysis
missing_counts = products.isnull().sum()
missing_pct = (missing_counts / len(products) * 100).round(2)

missing_summary = pd.DataFrame({
    'Column': products.columns,
    'Missing_Count': missing_counts.values,
    'Missing_Percentage': missing_pct.values
}).sort_values('Missing_Count', ascending=False)

print("\n[ANALYSIS] Missing Values (sorted by count):")
print(missing_summary.to_string(index=False))

In [ ]:
# Product dimension statistics
dimension_cols = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

dimension_stats = products[dimension_cols].describe().T

print("\n[ANALYSIS] Product Dimension Statistics:")
print(dimension_stats)

In [ ]:
# Product category analysis
category_counts = products['product_category_name'].value_counts().head(10)

print("\n[ANALYSIS] Top 10 Product Categories:")
print(category_counts)

print(f"\nTotal unique categories: {products['product_category_name'].nunique()}")
print(f"Products without category: {products['product_category_name'].isnull().sum():,}")

### 3.7 Sellers Dataset

In [ ]:
# Load sellers dataset
sellers = pd.read_csv(data_dir / 'olist_sellers_dataset.csv')

print("="*80)
print("SELLERS DATASET PROFILE")
print("="*80)

print(f"\n[DATA] Shape: {sellers.shape[0]:,} rows x {sellers.shape[1]} columns")

sellers.head(5)

In [ ]:
# Seller geographic distribution
state_counts = sellers['seller_state'].value_counts().head(10)
state_pct = (state_counts / len(sellers) * 100).round(2)

seller_geo_summary = pd.DataFrame({
    'State': state_counts.index,
    'Seller_Count': state_counts.values,
    'Percentage': state_pct.values
})

print("\n[ANALYSIS] Top 10 States by Seller Count:")
print(seller_geo_summary.to_string(index=False))

print(f"\nTotal states: {sellers['seller_state'].nunique()}")
print(f"Total cities: {sellers['seller_city'].nunique():,}")

---
## Step 4: Cross-Dataset Relationship Analysis

Now we'll analyze how datasets relate to each other using **vectorized merge operations**.

In [ ]:
# Analyze order_id relationship across datasets
print("="*80)
print("CROSS-DATASET RELATIONSHIP ANALYSIS")
print("="*80)

# Use NumPy set operations for efficient comparison
order_ids_orders = set(orders['order_id'].unique())
order_ids_items = set(order_items['order_id'].unique())
order_ids_payments = set(payments['order_id'].unique())
order_ids_reviews = set(reviews['order_id'].unique())

print("\n[ANALYSIS] Order ID Distribution Across Datasets:")
print(f"Orders table: {len(order_ids_orders):,} unique order_ids")
print(f"Order Items table: {len(order_ids_items):,} unique order_ids")
print(f"Payments table: {len(order_ids_payments):,} unique order_ids")
print(f"Reviews table: {len(order_ids_reviews):,} unique order_ids")

In [ ]:
# Referential integrity checks using set operations
items_not_in_orders = order_ids_items - order_ids_orders
payments_not_in_orders = order_ids_payments - order_ids_orders
reviews_not_in_orders = order_ids_reviews - order_ids_orders

print("\n[ANALYSIS] Referential Integrity:")
print(f"Order Items without matching Orders: {len(items_not_in_orders)}")
print(f"Payments without matching Orders: {len(payments_not_in_orders)}")
print(f"Reviews without matching Orders: {len(reviews_not_in_orders)}")

if len(items_not_in_orders) + len(payments_not_in_orders) + len(reviews_not_in_orders) == 0:
    print("\n[SUCCESS] All foreign keys have matching primary keys")
else:
    print("\n[WARNING] Some referential integrity issues detected")

In [ ]:
# Coverage analysis
orders_with_items = order_ids_orders & order_ids_items
orders_with_payments = order_ids_orders & order_ids_payments
orders_with_reviews = order_ids_orders & order_ids_reviews

coverage_summary = pd.DataFrame({
    'Dataset': ['Order Items', 'Payments', 'Reviews'],
    'Orders_Covered': [
        len(orders_with_items),
        len(orders_with_payments),
        len(orders_with_reviews)
    ],
    'Coverage_Pct': [
        round(len(orders_with_items) / len(order_ids_orders) * 100, 2),
        round(len(orders_with_payments) / len(order_ids_orders) * 100, 2),
        round(len(orders_with_reviews) / len(order_ids_orders) * 100, 2)
    ]
})

print("\n[ANALYSIS] Dataset Coverage (% of orders):")
print(coverage_summary.to_string(index=False))

---
## Step 5: Summary Statistics and Comparison

Create a comprehensive summary of all datasets.

In [ ]:
# Create comprehensive summary using list comprehension and vectorized operations
datasets = {
    'customers': customers,
    'orders': orders,
    'order_items': order_items,
    'payments': payments,
    'reviews': reviews,
    'products': products,
    'sellers': sellers
}

summary_data = []

for name, df in datasets.items():
    summary_data.append({
        'Dataset': name,
        'Rows': len(df),
        'Columns': len(df.columns),
        'Memory_MB': round(df.memory_usage(deep=True).sum() / (1024**2), 2),
        'Missing_Values': df.isnull().sum().sum(),
        'Duplicate_Rows': df.duplicated().sum(),
        'Numeric_Cols': len(df.select_dtypes(include=[np.number]).columns),
        'Object_Cols': len(df.select_dtypes(include=['object']).columns)
    })

summary_df = pd.DataFrame(summary_data)

print("="*80)
print("COMPREHENSIVE DATASET SUMMARY")
print("="*80)
print()
print(summary_df.to_string(index=False))

print(f"\n[SUMMARY] Total Records: {summary_df['Rows'].sum():,}")
print(f"[SUMMARY] Total Memory: {summary_df['Memory_MB'].sum():.2f} MB")
print(f"[SUMMARY] Total Missing Values: {summary_df['Missing_Values'].sum():,}")

---
## Step 6: Data Quality Issues Summary

Consolidate all identified data quality issues.

In [ ]:
# Create data quality issues report
quality_issues = []

# Check each dataset for missing values
for name, df in datasets.items():
    missing = df.isnull().sum()
    for col in missing[missing > 0].index:
        quality_issues.append({
            'Dataset': name,
            'Issue_Type': 'Missing Values',
            'Column': col,
            'Count': missing[col],
            'Percentage': round(missing[col] / len(df) * 100, 2)
        })
    
    # Check for duplicate rows
    dup_count = df.duplicated().sum()
    if dup_count > 0:
        quality_issues.append({
            'Dataset': name,
            'Issue_Type': 'Duplicate Rows',
            'Column': 'All',
            'Count': dup_count,
            'Percentage': round(dup_count / len(df) * 100, 2)
        })

quality_issues_df = pd.DataFrame(quality_issues)

print("="*80)
print("DATA QUALITY ISSUES SUMMARY")
print("="*80)
print()

if len(quality_issues_df) > 0:
    print(quality_issues_df.to_string(index=False))
    
    # Save to CSV
    quality_issues_df.to_csv(reports_dir / 'data_quality_issues.csv', index=False)
    print(f"\n[SUCCESS] Quality issues saved to: {reports_dir / 'data_quality_issues.csv'}")
else:
    print("[SUCCESS] No data quality issues detected!")

---
## Step 7: Save Profiling Results

Export summary statistics for documentation.

In [ ]:
# Save comprehensive summary
summary_df.to_csv(reports_dir / 'data_profile_summary.csv', index=False)
print(f"[SUCCESS] Dataset summary saved to: {reports_dir / 'data_profile_summary.csv'}")

# Create text report
report_path = reports_dir / 'notebook_profiling_report.txt'

with open(report_path, 'w') as f:
    f.write("="*80 + "\n")
    f.write("OLIST ANALYTICS PRO - JUPYTER NOTEBOOK PROFILING REPORT\n")
    f.write("="*80 + "\n")
    f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Datasets Analyzed: {len(datasets)}\n")
    f.write("="*80 + "\n\n")
    
    f.write("DATASET SUMMARY\n")
    f.write("-" * 80 + "\n")
    f.write(summary_df.to_string(index=False))
    f.write("\n\n")
    
    if len(quality_issues_df) > 0:
        f.write("DATA QUALITY ISSUES\n")
        f.write("-" * 80 + "\n")
        f.write(quality_issues_df.to_string(index=False))
        f.write("\n\n")
    
    f.write("KEY FINDINGS\n")
    f.write("-" * 80 + "\n")
    f.write(f"Total Records: {summary_df['Rows'].sum():,}\n")
    f.write(f"Total Memory Usage: {summary_df['Memory_MB'].sum():.2f} MB\n")
    f.write(f"Total Missing Values: {summary_df['Missing_Values'].sum():,}\n")
    f.write(f"Total Duplicate Rows: {summary_df['Duplicate_Rows'].sum():,}\n")

print(f"[SUCCESS] Text report saved to: {report_path}")

---
## Key Findings and Next Steps

### Data Quality Summary

**Strengths:**
- Complete transactional data across all core tables
- Valid primary keys identified
- Strong referential integrity

**Issues Identified:**
1. Missing delivery dates in orders table
2. Missing product categories
3. Sparse review comments
4. Need payment reconciliation

### Next Steps (Phase 2)

1. Investigate missing values by order status
2. Validate referential integrity
3. Handle missing product categories
4. Reconcile payment amounts
5. Document cleaning rules

---

**END OF PHASE 1**